# Khảo Sát & Đánh Giá Các Giải Pháp Text-to-Speech (TTS) Tiếng Việt

**Dự án:** ViMMRC Voice Q&A  
**Thành viên phụ trách:** Trương Hoàng Duy (`text-to-speech_Duy/`)  

Notebook này thực hiện:
1. Khảo sát danh sách giọng đọc tiếng Việt của Edge-TTS và gTTS.
2. Thử nghiệm bộ tiền xử lý và chuẩn hóa văn bản (Vietnamese Text Normalization).
3. Phát thử âm thanh trực tiếp trong Notebook (`IPython.display.Audio`).
4. Đo đạc & so sánh độ trễ (Latency/RTF) giữa các phương pháp.
5. Giới thiệu mô hình Deep Learning Local (`facebook/mms-tts-vie`) phục vụ giai đoạn nghiên cứu tiếp theo.

## 1. Cài đặt và Import thư viện

In [ ]:
%pip install -q edge-tts gTTS soundfile pydub

In [ ]:
import os
import sys
import time
import asyncio
from IPython.display import Audio, display

# Đảm bảo import được module từ thư mục hiện tại
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.append(current_dir)

from normalizer import normalize_vietnamese_text
from synthesize import synthesize
from tts_engine import EdgeTTSEngine

print("Đã import thành công các module!")

## 2. Danh sách các giọng đọc tiếng Việt hỗ trợ bởi Edge-TTS

In [ ]:
import nest_asyncio
nest_asyncio.apply()

voices = asyncio.run(EdgeTTSEngine.list_vietnamese_voices())
print(f"Tìm thấy {len(voices)} giọng đọc tiếng Việt:\n")
for v in voices:
    print(f"- Tên: {v['ShortName']:<25} | Giới tính: {v['Gender']:<8} | Mô tả: {v['FriendlyName']}")

## 3. Thử nghiệm bộ chuẩn hóa văn bản (Vietnamese Text Normalization)

Văn bản từ mô hình MRC hoặc LLM thường chứa Markdown, nhãn trắc nghiệm (A, B, C, D), số liệu, ngày tháng, và tỷ lệ phần trăm. Bộ chuẩn hóa giúp câu nói tự nhiên và phát âm chuẩn xác.

In [ ]:
sample_answers = [
    "**Đáp án đúng là:** (B). Ngày 02/09/1945, Chủ tịch Hồ Chí Minh đọc bản Tuyên ngôn Độc lập tại quảng trường Ba Đình.",
    "A. Khu bảo tồn có diện tích hơn 1.500 km2 với hơn 95.8% diện tích rừng nguyên sinh bao phủ.",
    "(D) Dự án cần nguồn ngân sách khoảng 500.000 USD, tương đương 12.500.000.000 VND.",
    "C: Vận tốc di chuyển đạt khoảng 80 km/h trong điều kiện thời tiết 38°C."
]

for i, text in enumerate(sample_answers, 1):
    norm_text = normalize_vietnamese_text(text)
    print(f"\n--- Mẫu {i} ---")
    print(f"Gốc      : {text}")
    print(f"Chuẩn hóa: {norm_text}")

## 4. Tổng hợp & Nghe thử âm thanh (Audio Playback)

In [ ]:
os.makedirs("outputs", exist_ok=True)
sample_text = "Đáp án B: Thủ đô của Việt Nam là Hà Nội, thành phố vì hòa bình được UNESCO công nhận."

# 1. Giọng nữ Hoài My
file_female = synthesize(sample_text, output_path="outputs/demo_hoaimy.mp3", voice="vi-VN-HoaiMyNeural")
print("Giọng nữ Hoài My:")
display(Audio(file_female))

# 2. Giọng nam Nam Minh
file_male = synthesize(sample_text, output_path="outputs/demo_namminh.mp3", voice="vi-VN-NamMinhNeural")
print("Giọng nam Nam Minh:")
display(Audio(file_male))

## 5. So sánh Benchmark độ trễ (Latency) giữa Edge-TTS và gTTS

In [ ]:
bench_text = "Trường Đại học Công nghệ Thông tin, Đại học Quốc gia Thành phố Hồ Chí Minh là trung tâm đào tạo hàng đầu."

# Benchmark Edge-TTS
t0 = time.time()
res_edge = synthesize(bench_text, output_path="outputs/bench_edge.mp3", engine="edge")
time_edge = time.time() - t0

# Benchmark gTTS
t0 = time.time()
res_gtts = synthesize(bench_text, output_path="outputs/bench_gtts.mp3", engine="gtts")
time_gtts = time.time() - t0

print(f"Thời gian tổng hợp:")
print(f"- Edge-TTS: {time_edge:.3f} giây (Dung lượng: {os.path.getsize(res_edge)} bytes)")
print(f"- gTTS    : {time_gtts:.3f} giây (Dung lượng: {os.path.getsize(res_gtts)} bytes)")

## 6. Hướng phát triển: Khảo sát mô hình Local Deep Learning (`facebook/mms-tts-vie`)

Đối với môi trường chạy hoàn toàn offline không cần kết nối mạng bên ngoài, Meta đã phát hành mô hình `facebook/mms-tts-vie` (Massively Multilingual Speech) dựa trên kiến trúc VITS:

```python
# Ví dụ triển khai mô hình offline bằng HuggingFace Transformers:
from transformers import VitsModel, AutoTokenizer
import torch
import soundfile as sf

model_id = "facebook/mms-tts-vie"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = VitsModel.from_pretrained(model_id)

inputs = tokenizer(normalize_vietnamese_text("Xin chào bạn"), return_tensors="pt")
with torch.no_grad():
    output = model(**inputs).waveform

sf.write("outputs/local_mms.wav", output.squeeze().cpu().numpy(), samplerate=model.config.sampling_rate)
```

Đây sẽ là nội dung nghiên cứu mở rộng trong giai đoạn tiếp theo của đồ án.